# 03 · QFT 与相位估计：把相位写进比特，再读出来

对应讲义实验 B · 约 45 分钟 · 前置：01 编程基础

量子傅里叶变换（QFT）把整数写进相位；相位估计（QPE）用逆 QFT 把相位读出来。QPE 是 Shor 算法和许多容错算法的核心子程序。

**学完本节，你将能够**
- 读懂 QFT 线路，并用经典的离散傅里叶变换验证它；
- 说清 QFT 与经典 FFT 的本质区别；
- 实现相位估计，破解一个“神秘量子门”的相位。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mindquantum import Circuit, UN, H, X, Power, PhaseShift, Measure, Simulator, qft
from mqlab import check

## 1. QFT 线路

`qft(range(n))` 生成 n 比特 QFT 线路：每个比特一个 H，加上 $n(n-1)/2$ 个受控相位门，末尾用 SWAP 交换比特顺序。

In [ ]:
qft(range(4)).svg()

QFT 把整数 x 写进相位。以 $|101\rangle$（x = 5）为例：输出的 8 个振幅大小完全相同，信息全在相位里。

In [ ]:
sim = Simulator('mqvector', 3)
sim.set_qs(np.eye(8)[5])                       # |101⟩
sim.apply_circuit(qft(range(3)))
out = sim.get_qs()
print('振幅的模：  ', np.abs(out).round(3))
print('相位（圈）：', (np.angle(out) / (2 * np.pi) % 1).round(3))

所有振幅的模都是 $1/\sqrt{8}$：直接测量只能得到均匀随机的比特串。要用上这些相位，还需要后续的干涉——这就是第 4 节相位估计要做的事。

## 2. 验证：QFT 就是离散傅里叶变换

调试量子程序的好习惯：用经典方法验证。QFT 对振幅做的正是离散傅里叶变换（DFT）。教科书把 q0 当作最高位，而 MindSpore Quantum 使用小端序，因此比较时要在前后各做一次比特反转。

In [ ]:
n = 4
rng = np.random.default_rng(7)
psi = rng.normal(size=2 ** n) + 1j * rng.normal(size=2 ** n)
psi /= np.linalg.norm(psi)                     # 随机量子态

sim = Simulator('mqvector', n)
sim.set_qs(psi)
sim.apply_circuit(qft(range(n)))               # 量子线路

rev = [int(f'{i:0{n}b}'[::-1], 2) for i in range(2 ** n)]   # 比特反转
ref = np.fft.ifft(psi[rev], norm='ortho')[rev]              # 经典 DFT
print(np.allclose(sim.get_qs(), ref))

### 练习 1 · 逆 QFT 的经典参考

补全 `inverse_qft_reference(psi)`：用 NumPy 计算 `qft(range(n)).hermitian()` 作用在 psi 上的结果。

In [ ]:
def inverse_qft_reference(psi):
    n = int(np.log2(len(psi)))
    # 练习 1
    return ...

check.inverse_qft_reference(inverse_qft_reference)

## 3. QFT 与 FFT：快在哪里，代价是什么

| | 经典 FFT | 量子 QFT |
|---|---|---|
| 运算量 | $O(n\,2^n)$ 次运算 | $O(n^2)$ 个量子门 |
| 结果 | $2^n$ 个系数全部可读 | 藏在振幅里，测量只得到一个比特串 |

QFT 不是“更快的 FFT”：它无法输出傅里叶系数。它的价值在于作为子程序，把相位信息转换成可以测量的比特串。

## 4. 挑战 · 用相位估计破解神秘量子门

神秘门 U 满足 $U|1\rangle = e^{2\pi i\varphi}|1\rangle$，求相位 φ。线路分四步（计数比特 $q_0 \ldots q_{n-1}$，目标比特 $q_n$）：

1. 计数比特全部作用 H；目标比特用 X 制备到本征态 $|1\rangle$；
2. 受控 $U^{2^k}$：**最上面的计数比特控制最高次幂**，即 $q_i$ 控制 $U^{2^{n-1-i}}$。`Power(U, m).on(目标, 控制)` 表示受控的 $U^m$；
3. 对计数比特作用逆 QFT：`qft(range(n)).hermitian()`；
4. 测量计数比特。

In [ ]:
def qpe(U, n):
    circ = Circuit()
    # 挑战：按上面四步搭建线路
    ...
    return circ

check.qpe(qpe)

通过检查后，破解神秘相位：采样 100 次，取出现最多的比特串。教科书的线路按大端序给出 φ 的二进制，所以读结果时要反转比特串。

In [ ]:
true_phi = 0.625
U = PhaseShift(2 * np.pi * true_phi)          # 神秘门
n = 3
circ = qpe(U, n)
if len(circ) == 0:
    print('请先完成挑战')
else:
    circ = circ.remove_measure() + UN(Measure(), n)
    res = Simulator('mqvector', n + 1).sampling(circ, shots=100, seed=1)
    bits = max(res.data, key=res.data.get)[::-1]      # 反转比特串
    print(res.data, '→ φ ≈', int(bits, 2) / 2 ** n)

## 5. 延伸：相位无法用 n 位二进制精确表示时

φ = 1/3 不能写成有限位的二进制小数。此时结果不再确定，而是集中在最接近的 $k/2^n$ 附近；计数比特越多，估计越精确。

In [ ]:
phi = 1 / 3
if len(qpe(U, 3)) == 0:
    print('请先完成挑战')
else:
    fig, axes = plt.subplots(1, 3, figsize=(12, 2.8), sharey=True)
    for ax, n in zip(axes, [3, 5, 7]):
        sim = Simulator('mqvector', n + 1)
        sim.apply_circuit(qpe(PhaseShift(2 * np.pi * phi), n).remove_measure())
        p = (np.abs(sim.get_qs()) ** 2).reshape(2, -1).sum(axis=0)   # 计数比特的分布
        est = [int(f'{i:0{n}b}'[::-1], 2) / 2 ** n for i in range(2 ** n)]
        ax.bar(est, p, width=0.7 / 2 ** n, color='#c0392b')
        ax.axvline(phi, color='0.5', ls='--', lw=1)
        ax.set_title(f'n = {n}')
        ax.set_xlabel('estimated phase')
    axes[0].set_ylabel('probability')
    plt.tight_layout()
    plt.show()

## 小结

- QFT 用 $O(n^2)$ 个门完成 $2^n$ 维的傅里叶变换，但结果在振幅里，无法直接读出。
- 相位估计用受控 $U^{2^k}$ 把相位写进计数比特，再用逆 QFT 读出；n 个计数比特给出 n 位二进制精度。
- Shor 算法分解大数，本质上就是对“模乘”门做相位估计。